<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L1_Text_Processing_Tokenization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L1: Text Processing & Tokenization - Foundation of LLMs

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 1 of 15  
**Book Reference:** Build a Large Language Model (From Scratch) - Sebastian Raschka

---

## 📚 Learning Objectives

By the end of this lesson, you will:
- Understand how LLMs process text data
- Learn the difference between characters and tokens
- Build a simple tokenizer from scratch
- Create vocabulary mappings
- Implement encoding and decoding functions

---

## 🧠 Concept: Why Tokenization?

**Tokenization** is the process of breaking text into smaller units called tokens.

### Why Do We Need It?
LLMs don't understand text directly - they work with numbers. Tokenization bridges this gap:

```
Text → Tokens → Numbers → LLM Processing → Numbers → Tokens → Text
```

### Characters vs Tokens
- **Character:** Single symbol (a, b, c, 1, 2, !)
- **Token:** Meaningful unit (word, subword, or punctuation)

**Example:**
```
Text: "Hello, world!"
Characters: ['H', 'e', 'l', 'l', 'o', ',', ' ', 'w', 'o', 'r', 'l', 'd', '!']
Tokens: ['Hello', ',', 'world', '!']
```

### Why Not Just Use Characters?
- Too many sequences (long context)
- Loses semantic meaning
- Inefficient for LLMs

---

In [ ]:
# Step 1: Import Required Libraries
import os
import urllib.request
import re

print("✅ Libraries imported successfully!")
print("📦 os: Operating system interactions")
print("📦 urllib: Download files from URLs")
print("📦 re: Regular expressions for pattern matching")

## 📖 Part 1: Loading Text Data

We'll use a sample text file to demonstrate tokenization. This is a short story that will serve as our training corpus.

---

In [ ]:
# Step 2: Download Sample Text File

# Check if file already exists
if not os.path.exists("the-verdict.txt"):
    print("📥 Downloading sample text file...")
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
    print("✅ File downloaded successfully!")
else:
    print("✅ File already exists!")

# Read the file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"\n📊 Text Statistics:")
print(f"Total characters: {len(raw_text):,}")
print(f"\n📝 First 200 characters:")
print(raw_text[:200])
print("...")

## 🔧 Part 2: Building a Simple Tokenizer

We'll create a tokenizer that:
1. Splits text on whitespace and punctuation
2. Keeps punctuation as separate tokens
3. Removes empty strings

### Regular Expression Pattern
```python
r'([,.:;?_!"()\']|--|\s)'
```

This pattern matches:
- Punctuation: `,.:;?_!"()\`
- Double dash: `--`
- Whitespace: `\s`

---

In [ ]:
# Step 3: Tokenize the Text

# Split on whitespace and punctuation, keeping delimiters
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)

# Remove empty strings and strip whitespace
preprocessed = [item.strip() for item in preprocessed if item.strip()]

print(f"📊 Tokenization Results:")
print(f"Total tokens: {len(preprocessed):,}")
print(f"\n🔍 First 30 tokens:")
print(preprocessed[:30])

# Compare with character count
print(f"\n💡 Insight:")
print(f"Characters: {len(raw_text):,}")
print(f"Tokens: {len(preprocessed):,}")
print(f"Compression ratio: {len(raw_text) / len(preprocessed):.2f}x")
print(f"\nTokens are ~{len(raw_text) / len(preprocessed):.1f}x more efficient than characters!")

## 📚 Part 3: Building a Vocabulary

A **vocabulary** is a mapping between tokens and unique integer IDs.

### Why Do We Need This?
- LLMs work with numbers, not strings
- Each unique token gets a unique ID
- Allows efficient lookup and processing

### Process:
1. Get all unique tokens
2. Sort them (for consistency)
3. Assign each token a unique integer ID

---

In [ ]:
# Step 4: Create Vocabulary

# Get unique tokens and sort them
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(f"📚 Vocabulary Statistics:")
print(f"Unique tokens (vocabulary size): {vocab_size:,}")
print(f"Total tokens in text: {len(preprocessed):,}")
print(f"Average token frequency: {len(preprocessed) / vocab_size:.2f}")

# Create token-to-ID mapping
vocab = {token: integer for integer, token in enumerate(all_words)}

print(f"\n🔍 First 20 vocabulary entries:")
for i, (token, idx) in enumerate(vocab.items()):
    print(f"  '{token}' → {idx}")
    if i >= 19:
        break

print(f"\n... and {vocab_size - 20} more tokens")

## 🔢 Part 4: Encoding - Text to Numbers

**Encoding** converts tokens to their integer IDs using the vocabulary.

```
Text → Tokens → IDs
"Hello world" → ["Hello", "world"] → [245, 892]
```

---

In [ ]:
# Step 5: Implement Encoding Function

def encode(text, vocab):
    """
    Encode text to token IDs.
    
    Args:
        text: Input text string
        vocab: Token-to-ID mapping dictionary
    
    Returns:
        List of token IDs
    """
    # Tokenize
    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
    tokens = [item.strip() for item in tokens if item.strip()]
    
    # Encode (skip unknown tokens)
    ids = [vocab[token] for token in tokens if token in vocab]
    
    return tokens, ids

# Test encoding
sample_text = "I had come to Chicago"

tokens, encoded_ids = encode(sample_text, vocab)

print(f"🔤 Original text: '{sample_text}'")
print(f"\n📝 Tokens: {tokens}")
print(f"\n🔢 Encoded IDs: {encoded_ids}")
print(f"\n📊 Encoding details:")
for token, idx in zip(tokens, encoded_ids):
    print(f"  '{token}' → {idx}")

## 🔠 Part 5: Decoding - Numbers to Text

**Decoding** converts token IDs back to text using a reverse vocabulary.

```
IDs → Tokens → Text
[245, 892] → ["Hello", "world"] → "Hello world"
```

---

In [ ]:
# Step 6: Implement Decoding Function

# Create reverse mapping (ID-to-token)
id2token = {idx: token for token, idx in vocab.items()}

def decode(ids, id2token):
    """
    Decode token IDs back to text.
    
    Args:
        ids: List of token IDs
        id2token: ID-to-token mapping dictionary
    
    Returns:
        Decoded text string
    """
    tokens = [id2token[idx] for idx in ids]
    text = ' '.join(tokens)
    return text

# Test decoding
decoded_text = decode(encoded_ids, id2token)

print(f"🔢 Encoded IDs: {encoded_ids}")
print(f"\n🔠 Decoded text: '{decoded_text}'")
print(f"\n✅ Original text: '{sample_text}'")
print(f"\n🎯 Match: {decoded_text == sample_text}")

## 🎨 Part 6: Visualizing the Process

Let's visualize the complete tokenization pipeline.

---

In [ ]:
# Step 7: Complete Pipeline Demonstration

def tokenization_pipeline(text, vocab, id2token):
    """
    Demonstrate complete tokenization pipeline.
    """
    print("="*60)
    print("TOKENIZATION PIPELINE")
    print("="*60)
    
    # Step 1: Original text
    print(f"\n1️⃣ Original Text:")
    print(f"   '{text}'")
    print(f"   Length: {len(text)} characters")
    
    # Step 2: Tokenization
    tokens, ids = encode(text, vocab)
    print(f"\n2️⃣ Tokenization:")
    print(f"   Tokens: {tokens}")
    print(f"   Count: {len(tokens)} tokens")
    
    # Step 3: Encoding
    print(f"\n3️⃣ Encoding (Token → ID):")
    for token, idx in zip(tokens, ids):
        print(f"   '{token}' → {idx}")
    print(f"   Encoded: {ids}")
    
    # Step 4: Decoding
    decoded = decode(ids, id2token)
    print(f"\n4️⃣ Decoding (ID → Token):")
    print(f"   Decoded: '{decoded}'")
    
    # Step 5: Verification
    print(f"\n5️⃣ Verification:")
    print(f"   Original: '{text}'")
    print(f"   Decoded:  '{decoded}'")
    print(f"   Match: {'✅ Yes' if decoded == text else '❌ No'}")
    
    print("\n" + "="*60)

# Test with different examples
examples = [
    "I had come to Chicago",
    "Hello, world!",
    "The king and queen are royal"
]

for example in examples:
    tokenization_pipeline(example, vocab, id2token)
    print("\n")

## 🎓 Key Takeaways

1. **Tokenization** breaks text into meaningful units (tokens)
2. **Tokens** are more efficient than characters (~4-5x compression)
3. **Vocabulary** maps tokens to unique integer IDs
4. **Encoding** converts text → tokens → IDs
5. **Decoding** converts IDs → tokens → text
6. This simple tokenizer is the foundation for understanding advanced tokenizers (BPE, WordPiece)

### Important Concepts
- **Vocabulary size:** Number of unique tokens
- **Token frequency:** How often each token appears
- **Unknown tokens:** Tokens not in vocabulary (handled in advanced tokenizers)

---

## 🔬 Exercises

1. **Modify the tokenizer** to handle contractions (e.g., "don't" → "do", "n't")
2. **Calculate token frequency** distribution in the text
3. **Handle unknown tokens** by adding a special `<UNK>` token
4. **Compare different splitting patterns** and their effect on vocabulary size
5. **Implement a character-level tokenizer** and compare efficiency

---

## ❓ Common Questions

**Q1: Why do we have 4,690 tokens from 20,479 characters?**  
A: Characters ≠ Tokens. We split on whitespace and punctuation, creating meaningful units. Each word and punctuation mark becomes a token.

**Q2: Why sort the vocabulary?**  
A: Sorting ensures consistency. The same text will always produce the same token IDs.

**Q3: What happens with words not in vocabulary?**  
A: In this simple tokenizer, they're skipped. Advanced tokenizers use subword tokenization to handle this.

**Q4: Is this how GPT tokenizes text?**  
A: No, GPT uses Byte Pair Encoding (BPE), which we'll learn in L8. This is a simplified version for learning.

---

## 📚 Additional Resources

- **Book:** "Build a Large Language Model (From Scratch)" - Sebastian Raschka (Chapter 2)
- **Paper:** "Neural Machine Translation of Rare Words with Subword Units" (BPE)
- **Tutorial:** [HuggingFace Tokenizers](https://huggingface.co/docs/tokenizers/)
- **Video:** [Tokenization Explained](https://www.youtube.com/watch?v=zduSFxRajkE)

---

## ➡️ Next Lesson

**L2: Transformer Pipelines** - Learn to use pre-trained models with HuggingFace

---

**Congratulations! You've completed your first lesson! 🎉**

*You now understand how LLMs process text at the most fundamental level.*